# Market-Entry Growth Case (C2 flagship)

Last-mile logistics operator entering e-commerce fulfillment. One decision:
**enter in-house, partner, or decline** - priced on cost-to-serve and the
partner rate, with the top-down funnel as the context block. Every number
regenerates offline via `python notebooks/run_notebook.py` and writes
`out/decision_numbers.json`.

In [ ]:
import sys, os, json
from pathlib import Path

# Resolve the project root so `model` and `data` import regardless of cwd.
try:
    ROOT = Path(__file__).resolve().parent.parent
except NameError:
    here = Path.cwd()
    ROOT = next((p for p in [here, *here.parents]
                 if (p / 'model').is_dir() and (p / 'data').is_dir()), here)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from data import Facts, OPERATORS, cross_check, measured_fields
from model import sizing, unit_economics, decision

f = Facts()

## 01 - Top-down sizing funnel

SAM -> serviceable -> capturable, each stage tagged. The USPS filed
commercial-package figure is the independent CONFIRMED anchor; the EST
top-down lands inside it.

In [ ]:
fl = sizing.funnel(f)
for s in fl["stages"]:
    print(f"{s['stage']:32s} {s['parcels_per_year']:>16,.0f}  [{s['confidence']}]  {s['note']}")
anchor = fl["anchor"]
print(f"anchor: {anchor['usps_commercial_parcels_billion']}B USPS commercial parcels ({anchor['confidence']})")
print(f"sam_billion={fl['sam_billion']:.3f}  capturable={fl['capturable_parcels']:,.0f}")

## 02 - Bottom-up operator benchmarks

Per-operator revenue/parcel and op margin from public filings - the
filings that anchor the top-down estimate. The adjacent-market entry
case is RRD (document -> e-comm fulfillment).

In [ ]:
bd = decision.bottom_up_benchmarks(f)
print(bd.to_string(index=False))

## 03 - Unit economics: the flip volume

In-house all-in cost/parcel is `cv + F/V`; the partner rate is the flat
`p`. In-house is cheaper exactly when `V >= F/(p - cv)` - the flip
volume. The 5-yr PV break-even is the same boundary with cost decay and
the committee discount rate; it is strictly tighter than Y1.

In [ ]:
tbl = unit_economics.contribution_table(f, [50_000, 100_000, 200_000, 500_000, 800_000, 1_000_000])
for r in tbl["volumes"]:
    tag = "BUILD" if r["in_house_wins"] else "PARTNER"
    print(f"{r['volume_per_year']:>10,.0f}/yr  in-house ${r['in_house_cost_pp']:.2f} vs partner ${r['partner_cost_pp']:.2f}  -> {tag}")
print()
print(f"flip volume (Y1): {tbl['flip_volume']:,.0f}/yr")
print(f"5-yr PV break-even: {tbl['pv_break_even']:,.0f}/yr  (decay is worth real PV)")
print(f"decline rate: ${tbl['decline_rate']:.2f}/parcel (partner margin at the floor)")

## 04 - Enter / partner / decline trigger

The trigger line at the launch wedge, and the 2-D cost x rate surface
the partner screenshots. Below the flip you partner; above it you build;
at the decline rate the whole entry is unprofitable.

In [ ]:
t = decision.trigger_line(f)
print(t["recommendation"] + ": " + t["line"])
print()
print(f"to ENTER: volume >= {t['flip_volume']:,.0f}/yr; below the decline rate ${t['decline_rate']:.2f} -> DECLINE")
df = decision.surface(f)
print()
print(f"surface: {len(df)} (cost, rate) cells ->")
print(df.head(12).to_string(index=False))

## 05 - Sanity cross-check (the decision is cost, not SAM)

Halve every market-size factor and the flip volume does not move - the
funnel sets headroom, not the trigger. Every pivot input moves the
boundary. That is the claim the decision page rests on, proven offline.

In [ ]:
sc = cross_check()
print("passed:", sc["passed"])
for r in sc["invariants_held"]:
    print(f"  invariant  {r['probe']:40s} moved={r['boundary_moved']} ok={r['as_expected']}")
for r in sc["pivot_inputs_move_boundary"]:
    print(f"  sensitivity {r['probe']:40s} moved={r['boundary_moved']} ok={r['as_expected']}")
hr = sizing.headroom(f)
print()
print(f"headroom: capturable {sizing.capturable_parcels(f):,.0f} vs flip {hr['flip_volume']:,.0f} -> {hr['headroom_multiple']:.0f}x of runway")
ca = sizing.cac_payback(f)
print(f"CAC ${ca['cac_per_customer']:.0f}/customer -> {ca['parcels_to_recover_cac_in_house']:.1f} parcels in-house vs {ca['parcels_to_recover_cac_partner']:.0f} partner")

## 06 - Write the decision numbers

The JSON that pins every figure quoted in the report and the README.
Gitignored; regenerated by the one-command run.

In [ ]:
p = decision.page(f)
row = p.iloc[0]
out = {
    "funnel": fl,
    "operators": json.loads(bd.to_json(orient="records")),
    "flip_volume": tbl["flip_volume"],
    "pv_break_even": tbl["pv_break_even"],
    "decline_rate": tbl["decline_rate"],
    "trigger_line": t,
    "surface_cells": len(df),
    "page": {k: row[k] for k in ["recommendation_at_wedge", "trigger_line",
                                 "flip_volume", "pv_break_even",
                                 "decline_rate", "partner_margin_pp",
                                 "pv_build_at_wedge", "pv_lease_at_wedge"]},
    "size_block": row["size"],
    "volume_grid": row["volume_grid"],
    "cross_check_passed": sc["passed"],
    "headroom_multiple": hr["headroom_multiple"],
    "cac_payback": ca,
}
os.makedirs(ROOT / "out", exist_ok=True)
with open(ROOT / "out" / "decision_numbers.json", "w") as fh:
    json.dump(out, fh, indent=2, default=float)
print("wrote out/decision_numbers.json")
print(f"flip {tbl['flip_volume']:,.0f} | pv_be {tbl['pv_break_even']:,.0f} | recommendation {t['recommendation']}")